# 13 · GNN

Template para `RentalPriceGNN`, con foco en construcción del grafo, evaluación comparable y análisis de qué aprende la red.

## Hipótesis del modelo

- El precio surge de atributos del inmueble y de la estructura relacional entre observaciones cercanas.
- La definición del grafo es parte del modelo y debe documentarse con tanto cuidado como los hiperparámetros.
- La interpretación se apoya más en ablations, embeddings y atención que en importancias clásicas.

In [159]:
from pathlib import Path
import sys
import pandas as pd
import geopandas as gpd
import seaborn as sns


PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from ml_core.models.gat_gcn_model import GraphAttentionGCN
from ml_core.preprocessing.knhs import KNHS
from sklearn.neighbors import BallTree
from ml_core.evaluation.modelEvaluator import regression_metrics
from ml_core.visualization.mapper import *

from sklearn.metrics import mean_squared_error, mean_absolute_error


OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "output" / "12_rf_kriging"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

## Datos base

In [160]:
DATA_PATH = PROJECT_ROOT / "data" / "splits"

train_raw = pd.read_csv(DATA_PATH / "arg_venta_data_train.csv")
gdf_train = gpd.GeoDataFrame(
    train_raw,
    geometry=gpd.points_from_xy(
        train_raw["longitud"],
        train_raw["latitud"]
    ),
    crs="EPSG:4326"
)

test_raw = pd.read_csv(DATA_PATH / "arg_venta_data_test.csv")
gdf_test = gpd.GeoDataFrame(
    test_raw,
    geometry=gpd.points_from_xy(
        test_raw["longitud"],
        test_raw["latitud"]
    ),
    crs="EPSG:4326"
)

val_raw = pd.read_csv(DATA_PATH / "arg_venta_data_val.csv")
gdf_val = gpd.GeoDataFrame(
    val_raw,
    geometry=gpd.points_from_xy(
        val_raw["longitud"],
        val_raw["latitud"]
    ),
    crs="EPSG:4326"
)



target_col = "log_precio"   
coord_cols = ["longitud", "latitud"]
feature_cols = [
    'area_m2_total',
    'area_m2_descubierta',
    'ambientes',
    'antiguedad',
    'expensas',
    'banos',
    'cocheras',
    'estado_num',
    'disposicion_Frente',
    'disposicion_Contrafrente',
    'disposicion_Lateral'
]

feature_cols_to_std = [
    'area_m2_total',
    'area_m2_descubierta',
    'ambientes',
    'antiguedad',
    'expensas',
]


## Definición del grafo

Dejá explícito:

- cómo conectás nodos
- si usás KNN, radio, barrio o una mezcla
- qué atributos van en `edge_attr`
- si el grafo se arma solo con train o con todo el dataset

In [161]:
X_train = gdf_train[feature_cols]
y_train = gdf_train[target_col]
coords_train = gdf_train[coord_cols].to_numpy()

X_test = gdf_test[feature_cols]
y_test = gdf_test[target_col]
coords_test = gdf_test[coord_cols].to_numpy()

X_val = gdf_val[feature_cols]
y_val = gdf_val[target_col]
coords_val = gdf_val[coord_cols].to_numpy()

In [162]:
# Hyperparámetros de grafo
k_neighbors = 15  # vecinos por nodo para construir aristas
radius_km = 3.0
bandwidth_km = 2.0  # para pesos locales (LGWR aproximada)

# Pesos locales por feature (LGWR aproximada sobre train) y proyección a test

def compute_local_feature_weights(X_df, y_array, coords_rad, bandwidth_km=2.0, k_neighbors=50):
    """Estima pesos por feature usando regresión local ponderada espacialmente."""
    from sklearn.neighbors import BallTree

    X = X_df.to_numpy()
    n, d = X.shape
    tree = BallTree(coords_rad, metric='haversine')
    weights = np.zeros((n, d), dtype=float)

    for i in range(n):
        dist, idx = tree.query(coords_rad[i:i+1], k=min(k_neighbors + 1, n))
        dist = dist.ravel() * 6371.0
        idx = idx.ravel()
        mask = dist > 0
        dist, idx = dist[mask], idx[mask]
        if len(idx) == 0:
            weights[i] = 1.0
            continue
        
        kernel = np.exp(-(dist ** 2) / (bandwidth_km ** 2))
        Xn = X[idx]
        wn = kernel[:, None]

        print(type(X))
        print(type(Xn))
        print(Xn.dtype if hasattr(Xn, "dtype") else "no dtype")

        A = Xn * wn
        AtX = A.T @ Xn + np.eye(d) * 1e-6  # regularización numérica
        AtY = A.T @ y_array[idx]
        beta = np.linalg.solve(AtX, AtY)
        weights[i] = np.abs(beta)

    return weights


weights_train = compute_local_feature_weights(
    X_train,
    y_train,
    coords_train,
    bandwidth_km=bandwidth_km,
    k_neighbors=k_neighbors,
)

# Proyectar pesos a test como promedio ponderado de vecinos de train
_tree_w = BallTree(coords_train, metric='haversine')
k_proj = min(k_neighbors, len(coords_train))
dist_proj, idx_proj = _tree_w.query(coords_test, k=k_proj)
kernel_proj = np.exp(-((dist_proj * 6371.0) ** 2) / (bandwidth_km ** 2))
kernel_proj = kernel_proj / (kernel_proj.sum(axis=1, keepdims=True) + 1e-9)
weights_test = (kernel_proj[..., None] * weights_train[idx_proj]).sum(axis=1)

# Columnas de pesos asociadas a cada feature
weight_cols = [f'w_{c}' for c in feature_cols]

######################################################## Y ESTO???? CHEQUEAR..#############################################33

# Proyectar pesos a val como promedio ponderado de vecinos de train
_tree_w = BallTree(coords_train, metric='haversine')
k_proj = min(k_neighbors, len(coords_train))
dist_proj, idx_proj = _tree_w.query(coords_val, k=k_proj)
kernel_proj = np.exp(-((dist_proj * 6371.0) ** 2) / (bandwidth_km ** 2))
kernel_proj = kernel_proj / (kernel_proj.sum(axis=1, keepdims=True) + 1e-9)
weights_val = (kernel_proj[..., None] * weights_train[idx_proj]).sum(axis=1)

# Columnas de pesos asociadas a cada feature
weight_cols = [f'w_{c}' for c in feature_cols]


# Construir grafo para train usando KNHS con distancia localmente pesada

graph_train = X_train.copy().reset_index(drop=True)
graph_train['lat_deg'] = np.rad2deg(coords_train[:, 0])
graph_train['lon_deg'] = np.rad2deg(coords_train[:, 1])
for col, vals in zip(weight_cols, weights_train.T):
    graph_train[col] = vals

############################################################################################################################

builder = KNHS(
    lat_col='lat_deg',
    lon_col='lon_deg',
    feature_cols=feature_cols,
    weight_cols=weight_cols,
    distance='local_weighted',
    radius_km=radius_km,
    k=k_neighbors,
    add_reverse=True,
)

edge_index_train, edge_attr_train = builder.build(graph_train)
print('Aristas train:', edge_index_train.shape[1])

# Grafo para test (inductivo) usando los pesos proyectados

graph_test = X_test.copy().reset_index(drop=True)
graph_test['lat_deg'] = np.rad2deg(coords_test[:, 0])
graph_test['lon_deg'] = np.rad2deg(coords_test[:, 1])
for col, vals in zip(weight_cols, weights_test.T):
    graph_test[col] = vals

edge_index_test, edge_attr_test = builder.build(graph_test)
print('Aristas test:', edge_index_test.shape[1])


# Grafo para val  usando los pesos proyectados

graph_val = X_val.copy().reset_index(drop=True)
graph_val['lat_deg'] = np.rad2deg(coords_val[:, 0])
graph_val['lon_deg'] = np.rad2deg(coords_val[:, 1])
for col, vals in zip(weight_cols, weights_val.T):
    graph_val[col] = vals

edge_index_val, edge_attr_val = builder.build(graph_val)
print('Aristas val:', edge_index_val.shape[1])



<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
float64
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


## Fine-tuning

ACordarse, el wrapper de models tiene ya un métodod e fine_tuning, deberíamos hacer que este choclo que definimos acá quede dentro de la clase correspondiente.

In [163]:
# Búsqueda más exhaustiva de hiperparámetros (grid)
from itertools import product

hidden_opts = [64, 96, 128]
head_opts = [2, 4]
dropout_opts = [0.05, 0.10]
lr_opts = [1e-3, 5e-4]

configs = [
    dict(hidden=h, num_heads=nh, dropout=do, lr=lr)
    for h, nh, do, lr in product(hidden_opts, head_opts, dropout_opts, lr_opts)
]

results = []
for cfg in configs:
    # saltar combinaciones donde hidden no divide en cabezas (por seguridad)
    if cfg['hidden'] % cfg['num_heads'] != 0:
        continue
    print("Entrenando config", cfg)
    model_t = GraphAttentionGCN(
        feature_names=feature_cols,
        edge_dim=2,
        hidden=cfg['hidden'],
        num_layers=2,
        num_heads=cfg['num_heads'],
        dropout=cfg['dropout'],
        lr=cfg['lr'],
        weight_decay=1e-4,
        patience=50,
    )
    _ = model_t.fit(
        X_train,
        y_train,
        edge_index=edge_index_train,
        edge_attr=edge_attr_train,
        epochs=600,
    )
    preds = model_t.predict(
        X_test,
        edge_index=edge_index_test,
        edge_attr=edge_attr_test,
    )
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    mape = np.mean(np.abs((np.exp(preds) - np.exp(y_test)) / (np.exp(y_test) + 1e-8))) * 100
    results.append({**cfg, "rmse": rmse, "mae": mae, "mape": mape})

results_df = pd.DataFrame(results).sort_values(["mae", "rmse"]).reset_index(drop=True)
best_config = results_df.iloc[0].to_dict()
print("Resultados (ordenado por MAE, RMSE):")
print(results_df)
print("Mejor config:", best_config)


Entrenando config {'hidden': 64, 'num_heads': 2, 'dropout': 0.05, 'lr': 0.001}
Entrenando config {'hidden': 64, 'num_heads': 2, 'dropout': 0.05, 'lr': 0.0005}
Entrenando config {'hidden': 64, 'num_heads': 2, 'dropout': 0.1, 'lr': 0.001}
Entrenando config {'hidden': 64, 'num_heads': 2, 'dropout': 0.1, 'lr': 0.0005}
Entrenando config {'hidden': 64, 'num_heads': 4, 'dropout': 0.05, 'lr': 0.001}
Entrenando config {'hidden': 64, 'num_heads': 4, 'dropout': 0.05, 'lr': 0.0005}
Entrenando config {'hidden': 64, 'num_heads': 4, 'dropout': 0.1, 'lr': 0.001}
Entrenando config {'hidden': 64, 'num_heads': 4, 'dropout': 0.1, 'lr': 0.0005}
Entrenando config {'hidden': 96, 'num_heads': 2, 'dropout': 0.05, 'lr': 0.001}
Entrenando config {'hidden': 96, 'num_heads': 2, 'dropout': 0.05, 'lr': 0.0005}
Entrenando config {'hidden': 96, 'num_heads': 2, 'dropout': 0.1, 'lr': 0.001}
Entrenando config {'hidden': 96, 'num_heads': 2, 'dropout': 0.1, 'lr': 0.0005}
Entrenando config {'hidden': 96, 'num_heads': 4, 'dr

## Entrenamiento

In [164]:
# Inicializar modelo con la mejor config encontrada

#################################3 MODIFICAR PARA PODER PASARLE UN DCT CON PARAMS COMO HACEMOS CON EL RESOT DE MODELS. ################33

edge_dim = 2  # distancia km + distancia de features (localmente pesada)
model = GraphAttentionGCN(
    feature_names=feature_cols,
    edge_dim=edge_dim,
    hidden=int(best_config['hidden']),
    num_layers=2,
    num_heads=int(best_config['num_heads']),
    dropout=float(best_config['dropout']),
    lr=float(best_config['lr']),
    weight_decay=1e-4,
    patience=50,
)



# Entrenamiento
_ = model.fit(
    X_train,
    y_train,
    edge_index=edge_index_train,
    edge_attr=edge_attr_train,
    epochs=6000,
)


Early stopping en epoch 1164, best loss=0.295877


## Evaluación global

In [165]:

X_val = gdf_val[feature_cols]
coords_val = gdf_val[coord_cols].to_numpy()
y_val = gdf_val[target_col]


y_pred_log = model.predict(
    X_val,
    edge_index=edge_index_val,
    edge_attr=edge_attr_val,
)

y_pred = np.exp(y_pred_log)
y_true = np.exp(y_val)

metrics = regression_metrics(y_val, y_pred)
metrics

{'rmse': 38019211601.32995,
 'mae': 781906628.5460061,
 'r2': -4.092376181437356e+21,
 'bias': -781906628.5460061,
 'median_abs_error': 112067.91611935226,
 'mape': 6377058466.589283}

## Diagnósticos específicos

Ideas para explorar:

- performance por zona
- sensibilidad a la definición del grafo
- ablations de features espaciales
- evolución de loss y early stopping

In [166]:
# history = pd.DataFrame(model.history_) if hasattr(model, "history_") else pd.DataFrame()
# history.tail()

## Embeddings / representaciones aprendidas

Si más adelante querés interpretación más rica, este bloque es un buen lugar para proyectar embeddings con UMAP/t-SNE y colorearlos por barrio, rango de precio o error.

In [167]:
# embeddings = None  # completar si expones el penultimo layer
# attention_summary = None  # completar si guardas pesos de atencion

## Export

In [168]:
# test_export = test_df[[target_col] + coord_cols].copy()
# test_export = test_export.rename(columns={target_col: "y_true"})
# test_export["y_pred"] = np.asarray(y_pred).reshape(-1)
# test_export["residual"] = test_export["y_true"] - test_export["y_pred"]
# test_export["split"] = "test"
# test_export.to_parquet(OUTPUT_DIR / "test_predictions.parquet", index=False)
# if embeddings is not None:
#     pd.DataFrame(embeddings).to_parquet(OUTPUT_DIR / "interpretability.parquet", index=False)
# (OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
# (OUTPUT_DIR / "run_config.json").write_text(json.dumps(gnn_config, indent=2, ensure_ascii=False))